# 03 — Pemodelan BiLSTM pada Data Baru (v2): 5 Strategi Penyeimbangan Kelas Multi-Seed

Notebook ini menguji 5 strategi penyeimbangan kelas pada model BiLSTM dengan evaluasi multi-seed (`42, 123, 456`):
1. **Natural Baseline** (Tanpa Penyeimbangan)
2. **Class Weight (CW)**
3. **Random Oversampling (ROS)**
4. **Random Undersampling (RUS)**
5. **SMOTE**

Data dievaluasi pada **Test Set Empiris Terkunci ($n = 1.730$)** dari `banjir_processed_v2.csv`.


In [ ]:
import os
import sys
from pathlib import Path

def resolve_path(filename):
    """Cari file secara rekursif di /kaggle/input (Kaggle) atau kandidat lokal."""
    # 1. Rekursif cari di /kaggle/input (menangani semua variasi mount Kaggle)
    if Path("/kaggle/input").exists():
        for root, _dirs, files in os.walk("/kaggle/input"):
            if filename in files:
                found = os.path.join(root, filename)
                print(f"[resolve_path] Ditemukan di Kaggle: {found}")
                return found
    # 2. Kandidat lokal workstation
    candidates = [
        Path(f"Data/processed/{filename}"),
        Path(f"Data/simulated/{filename}"),
        Path(f"Data/raw/{filename}"),
        Path(f"Data/{filename}"),
        Path(f"kamus/{filename}"),
        Path(f"Output/predictions/{filename}"),
        Path(f"../Data/processed/{filename}"),
        Path(f"../Data/simulated/{filename}"),
        Path(f"../Data/raw/{filename}"),
        Path(f"../kamus/{filename}"),
        Path(filename),
    ]
    for p in candidates:
        if p.exists():
            print(f"[resolve_path] Ditemukan lokal: {p}")
            return str(p)
    return filename


In [ ]:
import numpy as np
import pandas as pd
import random
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, recall_score
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

# Load Data
csv_path = resolve_path('banjir_processed_v2.csv')
df = pd.read_csv(csv_path)
col_text = 'processed_text_v2'
col_label = 'label'

tr_val, test_df = train_test_split(df, test_size=0.20, stratify=df[col_label], random_state=42)
train_df, val_df = train_test_split(tr_val, test_size=0.10, stratify=tr_val[col_label], random_state=42)

tok = Tokenizer(num_words=10000, oov_token='<OOV>')
tok.fit_on_texts(train_df[col_text].astype(str))

X_tr = pad_sequences(tok.texts_to_sequences(train_df[col_text].astype(str)), maxlen=50, padding='post', truncating='post')
X_va = pad_sequences(tok.texts_to_sequences(val_df[col_text].astype(str)), maxlen=50, padding='post', truncating='post')
X_te = pad_sequences(tok.texts_to_sequences(test_df[col_text].astype(str)), maxlen=50, padding='post', truncating='post')

y_tr = train_df[col_label].values
y_va = val_df[col_label].values
y_te = test_df[col_label].values

print(f'Train: {len(X_tr)} | Val: {len(X_va)} | Test: {len(X_te)}')


In [ ]:
def build_bilstm(vocab_size=10000, embedding_dim=128, units=64, dropout=0.3, max_len=50, lr=0.0001):
    model = Sequential([
        Embedding(vocab_size, embedding_dim, input_length=max_len),
        Bidirectional(LSTM(units)),
        Dropout(dropout),
        Dense(3, activation='softmax')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

print('BiLSTM builder siap.')


In [ ]:
# Multi-Seed Execution (Seeds: 42, 123, 456) untuk 5 Strategi Penyeimbangan
strategies = ['Baseline', 'Class Weight', 'ROS', 'RUS', 'SMOTE']
seeds = [42, 123, 456]
summary_records = []

for strat in strategies:
    f1_list, acc_list, rec_net_list = [], [], []
    for s in seeds:
        tf.random.set_seed(s)
        np.random.seed(s)
        
        X_train_res, y_train_res = X_tr.copy(), y_tr.copy()
        cw_dict = None
        
        if strat == 'Class Weight':
            cw_vals = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
            cw_dict = dict(enumerate(cw_vals))
        elif strat == 'ROS':
            ros = RandomOverSampler(random_state=s)
            X_train_res, y_train_res = ros.fit_resample(X_tr, y_tr)
        elif strat == 'RUS':
            rus = RandomUnderSampler(random_state=s)
            X_train_res, y_train_res = rus.fit_resample(X_tr, y_tr)
        elif strat == 'SMOTE':
            smote = SMOTE(random_state=s)
            X_train_res, y_train_res = smote.fit_resample(X_tr, y_tr)
            X_train_res = np.round(X_train_res).astype('int32')
            
        m = build_bilstm()
        es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
        m.fit(X_train_res, y_train_res, validation_data=(X_va, y_va), epochs=20, batch_size=16, class_weight=cw_dict, callbacks=[es], verbose=0)
        preds = np.argmax(m.predict(X_te, verbose=0), axis=1)
        
        acc_list.append(accuracy_score(y_te, preds))
        f1_list.append(f1_score(y_te, preds, average='macro', zero_division=0))
        rec_net_list.append(recall_score(y_te, preds, average=None, zero_division=0)[1])
        
    summary_records.append({
        'Strategi': strat,
        'Accuracy Mean (%)': round(np.mean(acc_list) * 100, 2),
        'Accuracy Std (%)': round(np.std(acc_list) * 100, 2),
        'Macro F1 Mean (%)': round(np.mean(f1_list) * 100, 2),
        'Macro F1 Std (%)': round(np.std(f1_list) * 100, 2),
        'Recall Netral Mean (%)': round(np.mean(rec_net_list) * 100, 2)
    })
    print(f'{strat} selesai: F1 = {np.mean(f1_list)*100:.2f}%  {np.std(f1_list)*100:.2f}%')


In [ ]:
# Rangkuman Hasil BiLSTM Multi-Seed
df_bilstm_summary = pd.DataFrame(summary_records)
print('=' * 75)
print('TABEL RANGKUMAN BiLSTM MULTI-SEED (DATA BARU V2 - 5 STRATEGI)')
print('=' * 75)
print(df_bilstm_summary.to_string(index=False))

out_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('Output/predictions')
out_dir.mkdir(parents=True, exist_ok=True)
df_bilstm_summary.to_csv(out_dir / 'results_bilstm_empiris_balancing.csv', index=False)
print(f'Tersimpan di: {out_dir / "results_bilstm_empiris_balancing.csv"}')
